Idea for Prompting:

You are an expert entity extraction system. Given an email thread, extract structured entities and return them as a single JSON object.

<instructions>

## Entity types to extract

### 1. people
All individuals who appear as senders, recipients, or are mentioned by name in the body text.

Fields:
- name          — Best available normalised full name
- aliases       — Array of ALL other names, identifiers, and email addresses used for this person across the thread (e.g. ["J", "jeffrey E.", "J Jep", "jeevacation@gmail.com"])
- roles         — Array of any applicable values: "sender", "recipient", "mentioned"
- email         — Primary email address if available, else null
- dedup_confidence — "high", "medium", or "low" (only present when this record merges two or more distinct identifiers; omit otherwise)

### 2. organisations
Companies, government agencies, media outlets, financial institutions, legal bodies, and other named groups.

Fields:
- name      — Organisation name as it appears in the text
- category  — One of: "media", "financial", "government", "legal", "other"
- context   — One-sentence note on how this organisation appears in the thread

### 3. dates_and_times
All temporal references from both email headers and body text.

Fields:
- raw        — Exact text as it appears in the source
- normalized — ISO 8601 format (YYYY-MM-DD or YYYY-MM-DDTHH:MM) if determinable; null otherwise
- type       — One of: "email_timestamp", "scheduled_event", "referenced_date"
- context    — Brief note on what this date refers to (e.g. "Timestamp of Message 2", "Meeting time mentioned in body")

### 4. action_items
Concrete tasks, follow-ups, or commitments — either explicitly stated or strongly implied.

Fields:
- description    — What needs to be done
- owner          — Name or identifier of the responsible party, or null if unclear
- due            — Date or time reference if mentioned, else null
- source_message — Which message this was extracted from (e.g. "Message 3")

</instructions>

<rules>

DEDUPLICATION
Consolidate the same person across all name and identifier variants into a single record. Use the most complete name available as `name` and list all other variants in `aliases`. Apply best judgement when variants are ambiguous — for example, "J" in a thread that also contains "jeffrey E." likely refers to the same person. When merging two or more distinct identifiers, set `dedup_confidence` to "high" (very confident), "medium" (likely but not certain), or "low" (plausible but uncertain).

TYPO AND OCR TOLERANCE
Treat names that differ only by a single character substitution, transposition, or obvious OCR corruption as the same person (e.g. "Thorbjon Jagland" and "Thorbjgn Jagland"; "gmaxl@ellmax.com" and "gmax1@ellmax.com"). List the corrupted variant in `aliases`.

BODY-REFERENCED PEOPLE
Include people mentioned in the body even if they never appear as senders or recipients. Set their role to "mentioned".

ACTION ITEMS
Only extract concrete commitments or explicit requests — things a person has said they will do, or directly asked another person to do. Do not extract vague opinions, observations, or implications.

UNKNOWN VALUES
Use null for any field whose value is not present in the text.

DO NOT INFER
Do not fabricate or infer any details not explicitly stated in the text.

</rules>

<output_format>
Return a single valid JSON object with exactly four top-level keys: "people", "organisations", "dates_and_times", "action_items". Each value is an array of objects conforming to the schemas above. Output only the JSON — no explanation, no markdown fencing.
</output_format>

<email_thread>
{{EMAIL_THREAD}}
</email_thread>

Extract entities from the email thread below. Return a single JSON object matching this schema exactly.

<schema>
{
  "people": [
    {
      "name": "Best available full name",
      "aliases": ["Every other identifier: nicknames, abbreviations, email addresses"],
      "roles": ["sender", "recipient", "mentioned"],
      "email": "primary email or null",
      "dedup_note": "Only present when merging ambiguous identifiers — explain reasoning"
    }
  ],
  "orgs": [
    {
      "name": "Organisation name",
      "type": "media | financial | government | legal | other",
      "context": "Why it appears in the thread (one sentence)"
    }
  ],
  "dates": [
    {
      "raw": "Verbatim from source",
      "iso": "YYYY-MM-DD or YYYY-MM-DDTHH:MM, or null if not determinable",
      "type": "email_sent | event | reference",
      "re": "What this date refers to"
    }
  ],
  "actions": [
    {
      "task": "Concrete commitment or explicit request",
      "owner": "Responsible person or null",
      "due": "When, or null",
      "msg": "Message N"
    }
  ]
}
</schema>

<rules>
1. DEDUP: One record per real person. Merge all name variants, abbreviations, and email addresses into aliases. When a merge is uncertain (e.g. a lone initial like "J"), include a dedup_note explaining your reasoning.
2. OCR/TYPO TOLERANCE: Names differing by one character (transposition, substitution, missing letter) are the same person. List the corrupted form in aliases.
3. BODY MENTIONS: People named only in email bodies get role "mentioned".
4. ACTIONS: Only extract tasks someone has committed to doing or explicitly asked another person to do. Opinions, observations, and forwarded content are not actions.
5. null for unknown fields. Do not infer or fabricate.
6. Output only valid JSON. No markdown fencing, no explanation.
</rules>
